In [0]:
-- ============================================================
-- GOLD 01: CONTEXTO ECONÓMICO POR PAÍS
-- ============================================================
-- Objetivo:
-- Integrar indicadores externos para complementar el análisis
-- financiero de KRAF.
--
-- Fuentes:
--   1. Inflación: indicador económico por país y año.
--   2. Riesgo soberano: rating crediticio del país.
--
-- La capa Gold integra información previamente preparada
-- en Silver para facilitar su consumo analítico.
-- ============================================================

CREATE OR REPLACE TABLE workspace.kraf_gold.contexto_economico_pais AS

SELECT
    i.country_code,
    i.country_name,
    i.year,
    i.inflation_rate,

    -- Rating soberano utilizado como referencia de riesgo país.
    r.rating_foreign,
    r.rating_foreign_outlook,
    r.rating_local,
    r.rating_local_outlook

FROM workspace.kraf_silver.global_inflation_silver AS i

LEFT JOIN workspace.kraf_silver.sovereign_credit_ratings_silver AS r
    ON UPPER(TRIM(i.country_name)) = UPPER(TRIM(r.sovereign))

WHERE i.year IS NOT NULL;

In [0]:
-- ============================================================
-- VALIDACIÓN
-- Verificamos la cantidad de registros generados en Gold.
-- ============================================================

SELECT COUNT(*) AS total_registros
FROM workspace.kraf_gold.contexto_economico_pais;

In [0]:
-- ============================================================
-- MUESTRA DE RESULTADOS
-- Revisamos algunos registros para comprobar la integración.
-- ============================================================

SELECT *
FROM workspace.kraf_gold.contexto_economico_pais
ORDER BY year DESC
LIMIT 10;

In [0]:
-- ============================================================
-- VALIDACIÓN DEL JOIN
-- Identificamos cuántos países tienen información de riesgo
-- soberano disponible y cuántos no encontraron coincidencia.
-- ============================================================

SELECT
    CASE
        WHEN rating_foreign IS NOT NULL THEN 'CON RIESGO'
        ELSE 'SIN COINCIDENCIA'
    END AS estado_riesgo,
    COUNT(*) AS total_registros

FROM workspace.kraf_gold.contexto_economico_pais

GROUP BY
    CASE
        WHEN rating_foreign IS NOT NULL THEN 'CON RIESGO'
        ELSE 'SIN COINCIDENCIA'
    END;

In [0]:
-- ============================================================
-- GOLD 02: CONTEXTO DE TIPO DE CAMBIO
-- ============================================================
-- Objetivo:
-- Preparar los indicadores cambiarios seleccionados para
-- Latinoamérica.
--
-- Se mantienen las cuatro monedas seleccionadas en Silver.
-- No se fuerza una relación con las transferencias KRAF,
-- porque la fuente no contiene una clave de país compatible.
-- ============================================================

CREATE OR REPLACE TABLE workspace.kraf_gold.contexto_tipo_cambio AS

SELECT
    fecha,
    brazilian_real,
    chilean_peso,
    colombian_peso,
    mexican_peso

FROM workspace.kraf_silver.currency_exchange_rates_silver

WHERE fecha IS NOT NULL;

In [0]:
-- ============================================================
-- VALIDACIÓN
-- Verificamos los registros disponibles en el contexto
-- de tipo de cambio.
-- ============================================================

SELECT COUNT(*) AS total_registros
FROM workspace.kraf_gold.contexto_tipo_cambio;

In [0]:
-- ============================================================
-- MUESTRA DE RESULTADOS
-- ============================================================

SELECT *
FROM workspace.kraf_gold.contexto_tipo_cambio
ORDER BY fecha DESC
LIMIT 10;

In [0]:
-- ============================================================
-- GOLD 03: CONTEXTO SOCIODEMOGRÁFICO DE EL SALVADOR
-- ============================================================
-- Objetivo:
-- Resumir la fuente sintética Nemotron para obtener contexto
-- sociodemográfico por departamento y área geográfica.
--
-- IMPORTANTE:
-- Estos perfiles son datos sintéticos y se utilizan únicamente
-- como contexto externo. No representan clientes reales de KRAF.
-- ============================================================

CREATE OR REPLACE TABLE workspace.kraf_gold.contexto_sociodemografico_sv AS

SELECT
    department,
    area,
    COUNT(*) AS total_personas,
    ROUND(AVG(age), 1) AS edad_promedio,
    MIN(age) AS edad_minima,
    MAX(age) AS edad_maxima

FROM workspace.kraf_silver.nemotron_personas_sv_silver

WHERE department IS NOT NULL
  AND area IS NOT NULL

GROUP BY
    department,
    area;

In [0]:
-- ============================================================
-- VALIDACIÓN
-- Revisamos los segmentos sociodemográficos generados.
-- ============================================================

SELECT COUNT(*) AS total_segmentos
FROM workspace.kraf_gold.contexto_sociodemografico_sv;

In [0]:
-- ============================================================
-- MUESTRA DE RESULTADOS
-- Visualizamos la distribución por departamento y área.
-- ============================================================

SELECT *
FROM workspace.kraf_gold.contexto_sociodemografico_sv
ORDER BY department, area;

In [0]:
-- ============================================================
-- GOLD 04: BASE ANALÍTICA DE TRANSFERENCIAS KRAF
-- ============================================================
-- Objetivo:
-- Preparar las transferencias para el análisis final,
-- incorporando la cuenta y el cliente asociado.
--
-- Esta tabla será la base para identificar movimientos
-- que se alejan del comportamiento habitual del canal.
-- ============================================================

SELECT
    t.transferencia_id,
    t.cuenta_id,
    t.entidad_financiera_id,
    t.tipo_transferencia_id,
    t.monto,
    t.fecha_hora,
    t.cuenta_destino,
    t.referencia,
    t.canal,
    t.estado

FROM workspace.kraf_silver.transferencia_silver AS t

LIMIT 10;

In [0]:
-- ============================================================
-- GOLD 04: TRANSFERENCIAS ATÍPICAS DE KRAF
-- ============================================================
-- Objetivo:
-- Identificar movimientos cuyo monto se encuentra muy por
-- encima del comportamiento promedio de su canal.
--
-- Regla utilizada:
-- Una transferencia se considera ATÍPICA cuando su monto
-- supera dos veces el promedio del canal.
--
-- Esta regla funciona como indicador analítico de alerta.
-- No constituye por sí sola una detección de fraude.
-- ============================================================

CREATE OR REPLACE TABLE workspace.kraf_gold.transferencias_riesgo AS

WITH promedio_canal AS (

    SELECT
        canal,
        AVG(monto) AS promedio_canal

    FROM workspace.kraf_silver.transferencia_silver

    GROUP BY canal
)

SELECT
    t.transferencia_id,
    t.cuenta_id,
    t.entidad_financiera_id,
    t.tipo_transferencia_id,
    t.monto,
    t.fecha_hora,
    t.cuenta_destino,
    t.referencia,
    t.canal,
    t.estado,

    -- Promedio histórico del canal utilizado.
    p.promedio_canal,

    -- Diferencia entre el monto de la transferencia
    -- y el promedio de su canal.
    t.monto - p.promedio_canal AS diferencia_promedio,

    -- Clasificación analítica de la transferencia.
    CASE
        WHEN t.monto > p.promedio_canal * 2
            THEN 'ATIPICA'
        ELSE 'NORMAL'
    END AS clasificacion

FROM workspace.kraf_silver.transferencia_silver AS t

LEFT JOIN promedio_canal AS p
    ON t.canal = p.canal;

In [0]:
-- ============================================================
-- MUESTRA DE TRANSFERENCIAS ATÍPICAS
-- ============================================================

SELECT
    transferencia_id,
    cuenta_id,
    monto,
    canal,
    promedio_canal,
    diferencia_promedio,
    clasificacion
FROM workspace.kraf_gold.transferencias_riesgo
WHERE clasificacion = 'ATIPICA'
ORDER BY monto DESC
LIMIT 10;

In [0]:
-- ============================================================
-- VALIDACIÓN FINAL DEL RIESGO TRANSACCIONAL
-- Contabilizamos las transferencias según su clasificación.
-- ============================================================

SELECT
    clasificacion,
    COUNT(*) AS total_transferencias
FROM workspace.kraf_gold.transferencias_riesgo
GROUP BY clasificacion
ORDER BY clasificacion;

In [0]:
SELECT 
    'contexto_economico_pais' AS tabla,
    COUNT(*) AS registros
FROM workspace.kraf_gold.contexto_economico_pais

UNION ALL

SELECT 
    'contexto_tipo_cambio',
    COUNT(*)
FROM workspace.kraf_gold.contexto_tipo_cambio

UNION ALL

SELECT 
    'contexto_sociodemografico_sv',
    COUNT(*)
FROM workspace.kraf_gold.contexto_sociodemografico_sv

UNION ALL

SELECT 
    'transferencias_riesgo',
    COUNT(*)
FROM workspace.kraf_gold.transferencias_riesgo;